# 16.9 知识图谱嵌入 / Knowledge Graph Embeddings (TransE & RotatE)

**中文**：**知识图谱(Knowledge Graph, KG)** 是一种特殊的图——边是**有类型**的，每条边是一个 **(头实体, 关系, 尾实体)** 三元组，如 (北京, 首都, 中国)、(爱因斯坦, 出生于, 德国)。KG 是搜索(Google 知识面板)、问答、推荐、大模型外挂知识库的核心。本节学 **KG 嵌入**:把实体和关系都变成向量，用几何运算表达"关系"，从而做 **知识补全(link prediction)**——预测缺失的三元组。这是 Part 16 的收官。
**English**: A **Knowledge Graph (KG)** is a special graph — edges are **typed**, each a **(head, relation, tail)** triple like (Beijing, capital_of, China) or (Einstein, born_in, Germany). KGs power search (Google's knowledge panel), question answering, recommendation, and LLM knowledge bases. This section covers **KG embeddings**: turn entities and relations into vectors and express "relations" as geometric operations, enabling **link prediction (KG completion)** — predicting missing triples. This closes Part 16.

---

**中文**：**TransE(2013)** 的思想极简优雅——**把关系看成向量空间里的"平移(translation)"**:
**English**: **TransE (2013)** is minimalist and elegant — **treat a relation as a "translation" in vector space**:

$$\mathbf h + \mathbf r \approx \mathbf t \quad\Longrightarrow\quad \text{打分/score } f(h,r,t) = -\|\mathbf h + \mathbf r - \mathbf t\|$$

**中文**：即"头实体向量 + 关系向量 ≈ 尾实体向量"。比如 $\mathbf{北京}+\mathbf{首都}\approx\mathbf{中国}$，而且理想情况下 $\mathbf{巴黎}+\mathbf{首都}\approx\mathbf{法国}$ ——**同一个关系是同一个平移向量**(和 Word2Vec 的 king−man+woman≈queen 一脉相承)。训练用**间隔排序损失(margin ranking loss)** + 负采样:让真三元组的分高于随机破坏的假三元组。
**English**: i.e. "head vector + relation vector ≈ tail vector." E.g. $\mathbf{Beijing}+\mathbf{capital}\approx\mathbf{China}$, and ideally $\mathbf{Paris}+\mathbf{capital}\approx\mathbf{France}$ — **the same relation is the same translation** (kin to Word2Vec's king−man+woman≈queen). Trained with a **margin ranking loss** + negative sampling: true triples score higher than randomly corrupted false ones.

**中文**：但 TransE 有个**致命的数学缺陷——无法建模对称关系**。若关系 $r$ 对称(如"配偶":(a,配偶,b) 与 (b,配偶,a) 都成立)，则要求 $\mathbf a+\mathbf r\approx\mathbf b$ **且** $\mathbf b+\mathbf r\approx\mathbf a$，两式相加得 $2\mathbf r\approx 0$，即 $\mathbf r\approx 0$——关系向量被迫坍缩成零，模型再也分不清谁和谁是配偶。
**English**: But TransE has a **fatal mathematical flaw — it cannot model symmetric relations**. If $r$ is symmetric (e.g. "spouse": both (a,spouse,b) and (b,spouse,a) hold), then it needs $\mathbf a+\mathbf r\approx\mathbf b$ **and** $\mathbf b+\mathbf r\approx\mathbf a$; adding gives $2\mathbf r\approx 0$, i.e. $\mathbf r\approx 0$ — the relation vector collapses to zero and the model can no longer tell who is whose spouse.

**中文**：**RotatE(2019)** 的解法漂亮——**把关系看成复数空间里的"旋转(rotation)"**:
**English**: **RotatE (2019)** fixes this beautifully — **treat a relation as a "rotation" in complex space**:

$$\mathbf t \approx \mathbf h \circ \mathbf r,\quad |r_i|=1\ (\text{即 } r_i=e^{i\theta_i}) \quad\Longrightarrow\quad f(h,r,t)=-\|\mathbf h\circ\mathbf r - \mathbf t\|$$

**中文**：每个实体是复向量，关系是**逐元素的单位复数(纯相位旋转)**$r_i=e^{i\theta_i}$，$\mathbf h\circ\mathbf r$ 是复数逐元素相乘(=对 $\mathbf h$ 的每一维旋转 $\theta_i$)。关键:**对称关系 = 旋转 180°($\theta=\pi$)**——转两次回到原点($e^{i\pi}\cdot e^{i\pi}=1$)，天然满足对称！RotatE 能同时表达**对称、反对称、逆关系、组合关系**四种模式，表达力远超 TransE。
**English**: Each entity is a complex vector, each relation is an **elementwise unit complex number (pure phase rotation)** $r_i=e^{i\theta_i}$, and $\mathbf h\circ\mathbf r$ is elementwise complex multiplication (= rotating each dim of $\mathbf h$ by $\theta_i$). Key: **a symmetric relation = a 180° rotation ($\theta=\pi$)** — applying it twice returns to the start ($e^{i\pi}\cdot e^{i\pi}=1$), so symmetry is satisfied by construction! RotatE can express **symmetry, antisymmetry, inversion, and composition** — far more expressive than TransE.

> 💡 **面试速查 / Interview cheat-sheet（★★★ KG 必考）**
> **中文**：KG=(头,关系,尾)三元组; KG 嵌入把实体/关系变向量做**链接预测(补全)**。**TransE**:关系=平移 $h{+}r{\approx}t$, 简单高效, 但**建模不了对称关系**(推出 r≈0)、也难处理 1-to-N。**RotatE**:关系=复空间旋转 $t{\approx}h\circ r,|r|{=}1$, 能表达**对称(转π)/反对称/逆/组合**四种模式。其他:DistMult(双线性, 只能对称)、ComplEx(复数双线性, 补上反对称)。评估用**过滤后的 MRR / Hits@k**(排名真尾实体, 屏蔽其它已知真三元组)。应用:搜索、问答、推荐(用户-物品-属性 KG)、药物-靶点、给 LLM 补事实。
> **English**: KG = (head, relation, tail) triples; KG embeddings vectorize entities/relations for **link prediction (completion)**. **TransE**: relation = translation $h{+}r{\approx}t$, simple and efficient, but **can't model symmetric relations** (forces r≈0) and struggles with 1-to-N. **RotatE**: relation = rotation in complex space $t{\approx}h\circ r,|r|{=}1$, expressing **symmetry (rotate π)/antisymmetry/inversion/composition**. Others: DistMult (bilinear, symmetric-only), ComplEx (complex bilinear, adds antisymmetry). Evaluate with **filtered MRR / Hits@k** (rank the true tail, masking other known true triples). Uses: search, QA, recommendation (user-item-attribute KG), drug-target, factual grounding for LLMs.


In [ ]:

# ============================================================
# 合成知识图谱:两个对称关系(RotatE 的主场, TransE 的软肋)
# Synthetic KG: two SYMMETRIC relations (RotatE's strength, TransE's weakness)
# 中文:每个实体在两个对称关系里各有一个不同的伙伴(完美匹配), 这样 TransE 无法靠"把两实体嵌成同一点"
#       来偷懒(因为另一关系要求它和别人也重合), 于是对称建模的缺陷会暴露在链接预测上。
# English: each entity has one distinct partner in each of two symmetric relations (perfect matchings),
#          so TransE can't cheat by collapsing paired entities (another relation forbids it) — exposing
#          its symmetric-modeling flaw in link prediction.
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0); rng=np.random.default_rng(0)
NE=200; NR=2
def matching(seed):                                          # 随机完美匹配 / a random perfect matching
    p=np.random.default_rng(seed).permutation(NE); return [(int(p[i]),int(p[i+1])) for i in range(0,NE-1,2)]
triples=[]
for a,b in matching(1): triples += [(a,0,b),(b,0,a)]         # 关系0:对称 / relation 0 symmetric
for a,b in matching(2): triples += [(a,1,b),(b,1,a)]         # 关系1:对称 / relation 1 symmetric
trip=np.array(sorted(set(triples))); rng.shuffle(trip)
train=[]; test=[]
for r in [0,1]:                                             # 每个关系各切 80/20 / split per relation
    sub=trip[trip[:,1]==r]; k=int(len(sub)*0.8); train+=list(sub[:k]); test+=list(sub[k:])
train=np.array(train); test=np.array(test); trueset=set(map(tuple,trip))
print(f"实体 {NE}, 关系 {NR}(均对称), 三元组 {len(trip)} (训练 {len(train)}, 测试 {len(test)})")


**中文**：从零实现 **TransE** 和 **RotatE**，都用 margin ranking loss + 负采样(随机替换尾实体造假三元组)。评估用**过滤后的链接预测**:对每个测试三元组 (h,r,?)，让模型给所有候选尾实体打分、看真尾实体排第几，屏蔽掉其它已知为真的三元组(filtered 设定)。指标 **MRR(平均倒数排名)** 和 **Hits@10**。
**English**: Implement **TransE** and **RotatE** from scratch, both with margin ranking loss + negative sampling (corrupt the tail). Evaluate with **filtered link prediction**: for each test (h,r,?), score all candidate tails and see where the true tail ranks, masking other known-true triples (the filtered setting). Metrics: **MRR (mean reciprocal rank)** and **Hits@10**.


In [ ]:

# ============================================================
# 从零实现 TransE 与 RotatE / TransE and RotatE from scratch
# ============================================================
class TransE(nn.Module):
    def __init__(s,d=32):
        super().__init__(); s.e=nn.Embedding(NE,d); s.r=nn.Embedding(NR,d)
        nn.init.uniform_(s.e.weight,-0.1,0.1); nn.init.uniform_(s.r.weight,-0.1,0.1)
    def dist(s,h,r,t): return (s.e(h)+s.r(r)-s.e(t)).norm(2,-1)               # ||h+r-t||
    def score_all(s,h,r):                                                     # 对所有候选尾实体打分 / score all tails
        hr=s.e(h)+s.r(r); return -(hr.unsqueeze(1)-s.e.weight.unsqueeze(0)).norm(2,-1)

class RotatE(nn.Module):
    def __init__(s,d=32):
        super().__init__(); s.d=d; s.e=nn.Embedding(NE,2*d); s.rp=nn.Embedding(NR,d)   # 实体复向量 / 关系相位
        nn.init.uniform_(s.e.weight,-0.1,0.1); nn.init.uniform_(s.rp.weight,-3.1416,3.1416)
    def _rot(s,h,r):                                                          # 复数逐元素旋转 h∘e^{iθ}
        hr,hi=h[...,:s.d],h[...,s.d:]; th=s.rp(r)
        return hr*torch.cos(th)-hi*torch.sin(th), hr*torch.sin(th)+hi*torch.cos(th)
    def dist(s,h,r,t):
        ar,ai=s._rot(s.e(h),r); tr,ti=s.e(t)[...,:s.d],s.e(t)[...,s.d:]
        return ((ar-tr)**2+(ai-ti)**2).sqrt().sum(-1)
    def score_all(s,h,r):
        ar,ai=s._rot(s.e(h),r); E=s.e.weight; tr,ti=E[:,:s.d],E[:,s.d:]
        return -(((ar.unsqueeze(1)-tr.unsqueeze(0))**2+(ai.unsqueeze(1)-ti.unsqueeze(0))**2).sqrt().sum(-1))

def train_kge(M, margin=6.0, epochs=400, bs=256):
    torch.manual_seed(0); m=M(); opt=torch.optim.Adam(m.parameters(),lr=0.01); Tr=torch.tensor(train)
    for ep in range(epochs):
        for b in range(0,len(Tr),bs):
            ba=Tr[torch.randperm(len(Tr))[b:b+bs]]; h,r,t=ba[:,0],ba[:,1],ba[:,2]
            tn=torch.randint(0,NE,(len(ba),))                                 # 负采样:替换尾实体 / corrupt tail
            loss=F.relu(margin + m.dist(h,r,t) - m.dist(h,r,tn)).mean()       # 间隔排序损失 / margin ranking
            opt.zero_grad(); loss.backward(); opt.step()
    return m

def link_pred(m):                                                            # 过滤后的 MRR / Hits@k + 排名数组
    ranks=[]
    with torch.no_grad():
        for h,r,t in test:
            s=m.score_all(torch.tensor([h]),torch.tensor([r]))[0]
            for tt in range(NE):
                if tt!=t and (h,r,tt) in trueset: s[tt]=-1e9                  # 过滤其它真三元组 / filtered
            ranks.append((s>s[t]).sum().item()+1)                            # 真尾实体的排名 / rank of true tail
    ranks=np.array(ranks); return np.mean(1/ranks), np.mean(ranks<=10), np.mean(ranks<=1), ranks

mt=train_kge(TransE); mr=train_kge(RotatE)
tr_mrr,tr_h10,tr_h1,tr_ranks=link_pred(mt); ro_mrr,ro_h10,ro_h1,ro_ranks=link_pred(mr)
print(f"{'模型/model':<10}{'MRR':>8}{'Hits@1':>9}{'Hits@10':>9}")
print(f"{'TransE':<10}{tr_mrr:>8.3f}{tr_h1:>9.3f}{tr_h10:>9.3f}")
print(f"{'RotatE':<10}{ro_mrr:>8.3f}{ro_h1:>9.3f}{ro_h10:>9.3f}")


**中文**：在这两个**对称关系**上，**RotatE 的 MRR 约为 TransE 的 2 倍，而 Hits@1 更是天壤之别**(TransE≈0，RotatE≈0.56)——也就是说，TransE 在对称关系上**几乎从来没能把真尾实体排到第 1 名**，因为它无法用一个平移同时满足 $a+r\approx b$ 和 $b+r\approx a$。下面直接**看证据**:两模型对"真尾实体排名"的分布(图②)，以及 RotatE 学到的关系相位是否集中在 $\pm\pi$(180° 旋转=自逆=对称)附近(图③)。
**English**: On these two **symmetric relations**, **RotatE's MRR is about 2× TransE's, and Hits@1 is a world apart** (TransE≈0, RotatE≈0.56) — that is, TransE **almost never ranks the true tail #1** on symmetric relations, because it cannot satisfy both $a+r\approx b$ and $b+r\approx a$ with one translation. Now the **direct evidence**: the two models' distributions of "true-tail rank" (plot ②) and whether RotatE's learned phases cluster near $\pm\pi$ (a 180° rotation = self-inverse = symmetric, plot ③).


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,5))
# ① MRR/Hits 对比 / metric comparison
labels=["MRR","Hits@1","Hits@10"]; xx=np.arange(3); w=0.35
ax[0].bar(xx-w/2,[tr_mrr,tr_h1,tr_h10],w,label="TransE",color="#C44E52")
ax[0].bar(xx+w/2,[ro_mrr,ro_h1,ro_h10],w,label="RotatE",color="#4C72B0")
ax[0].set_xticks(xx); ax[0].set_xticklabels(labels); ax[0].set_title("对称关系链接预测:RotatE 胜 / RotatE wins on symmetric"); ax[0].legend()
# ② 真尾实体排名分布:RotatE 集中在第1名, TransE 分散 / rank distribution of the true tail
bins=[1,2,4,8,16,32,64,200]
ax[1].hist(tr_ranks,bins=bins,alpha=0.6,label="TransE",color="#C44E52")
ax[1].hist(ro_ranks,bins=bins,alpha=0.6,label="RotatE",color="#4C72B0")
ax[1].set_xscale("log"); ax[1].set_title("真尾实体的排名分布 / rank of the true tail\n(越靠左越好, RotatE 多在第1名)")
ax[1].set_xlabel("rank (log)"); ax[1].set_ylabel("#test triples"); ax[1].legend()
# ③ RotatE 学到的相位分布:集中在 ±π / RotatE phase histogram
ph=mr.rp.weight.detach().numpy().ravel()
ax[2].hist(ph,bins=40,color="#55A868",edgecolor="white")
for x in [-np.pi,np.pi]: ax[2].axvline(x,ls="--",color="red")
ax[2].axvline(0,ls=":",color="gray")
ax[2].set_title("RotatE 关系相位 θ 分布 / phase distribution"); ax[2].set_xlabel("θ (红线=±π=对称旋转)")
plt.tight_layout(); plt.savefig("/tmp/g09_viz.png",dpi=80); plt.show()
frac_pi=(np.abs(ph)>2.4).mean()
print(f"TransE Hits@1={tr_h1:.3f} (对称关系上几乎从不把真尾排第1) vs RotatE Hits@1={ro_h1:.3f}")
print(f"RotatE 相位落在 |θ|>2.4(靠近±π) 的比例 / frac near ±π = {frac_pi:.2f}")


**中文**：诚实解读：
**English**: Honest takeaways:

**中文**：
1. **RotatE 在对称关系上明显胜出**:MRR 约为 TransE 的 2 倍，Hits@1 从 ~0 跃升到 ~0.56。原因就是那个简单的数学事实——对称关系要求 $2\mathbf r\approx0$，TransE 无法用一个平移把 $a\to b$ 又把 $b\to a$，于是真尾实体排名分散、极少排第 1(图②); 而 RotatE 用"旋转 180°"这一自逆操作天然表达对称(图③:相位出现向 $\pm\pi$ 聚集的倾向)。
2. **这不代表 TransE 没用**:TransE 简单、高效、在**功能性/层级性**关系(如"出生地""上级")上表现很好, 至今是强基线。**每种 KGE 模型都有它擅长和不擅长的关系模式**——这正是面试爱考的点(下表)。
3. **诚实的规模说明**:本节用小型合成 KG 是为了**干净地隔离"对称关系"这一机制**。在真实大规模基准(FB15k-237、WN18RR)上, RotatE 的整体 MRR(~0.34/~0.48)确实稳定高于 TransE(~0.29/~0.23), 但需要大数据、自对抗负采样等工程细节才能完全复现——机制我们已讲透, 数字规模化交给工业实现。

| 关系模式 / pattern | TransE | DistMult | RotatE |
|---|---|---|---|
| 对称 symmetric | ✗ | ✓ | ✓ |
| 反对称 antisymmetric | ✓ | ✗ | ✓ |
| 逆关系 inversion | ✓ | ✗ | ✓ |
| 组合 composition | ✓ | ✗ | ✓ |

**English**:
1. **RotatE clearly wins on symmetric relations**: MRR ~2× TransE, and Hits@1 jumps from ~0 to ~0.56. The cause is that simple math fact — a symmetric relation demands $2\mathbf r\approx0$, and TransE cannot map $a\to b$ and $b\to a$ with one translation, so the true tail's rank scatters and is rarely #1 (plot ②); RotatE expresses symmetry naturally via the self-inverse "180° rotation" (plot ③: phases show a tendency to cluster near $\pm\pi$).
2. **This doesn't make TransE useless**: TransE is simple, efficient, and strong on **functional/hierarchical** relations (e.g. "born_in," "manager_of"), still a solid baseline. **Every KGE model is good at some relation patterns and bad at others** — a favorite interview point (table).
3. **An honest note on scale**: we use a small synthetic KG to **cleanly isolate the "symmetric relation" mechanism**. On real large benchmarks (FB15k-237, WN18RR), RotatE's overall MRR (~0.34/~0.48) is indeed consistently above TransE (~0.29/~0.23), but fully reproducing that needs big data, self-adversarial negative sampling, etc. — we've taught the mechanism; the scaled numbers are for industrial implementations.

> 💼 **实战视角 / Practical angle**
> **中文**:KG 嵌入用于:① **知识补全**(填缺失事实, 如"这个药可能治哪种病"); ② **推荐**(把用户-物品-属性建成 KG, 阿里/美团都用 KGE 增强召回与可解释); ③ **搜索/问答**(实体链接、关系推理); ④ **给大模型注入结构化知识**、减少幻觉(RAG over KG)。选型:小而全用 TransE/DistMult 起步; 关系模式复杂用 RotatE/ComplEx; 超大 KG 用 PyKEEN/DGL-KE 等库 + 分布式。面试金句:*"TransE 关系是平移、简单但建模不了对称; RotatE 关系是复空间旋转、能表达对称/反对称/逆/组合; 评估用过滤后的 MRR/Hits@k。"*
> **English**: KG embeddings power: ① **knowledge completion** (fill missing facts, e.g. "which disease might this drug treat"); ② **recommendation** (model user-item-attribute as a KG; Alibaba/Meituan use KGE to boost recall and explainability); ③ **search/QA** (entity linking, relational reasoning); ④ **injecting structured knowledge into LLMs**, reducing hallucination (RAG over KG). Selection: start with TransE/DistMult for small/simple; use RotatE/ComplEx for complex relation patterns; huge KGs use libraries (PyKEEN/DGL-KE) + distributed. Interview line: *"TransE's relation is a translation — simple but can't model symmetry; RotatE's relation is a rotation in complex space — expresses symmetry/antisymmetry/inversion/composition; evaluate with filtered MRR/Hits@k."*

---
### 小结 / Summary
- **中文**:KG=(头,关系,尾)三元组; KGE 把实体/关系变向量做链接预测(补全)。
- **English**: KG = (head, relation, tail) triples; KGE vectorizes entities/relations for link prediction (completion).
- **中文**:TransE 关系=平移(h+r≈t), 简单但建模不了对称(推出 r≈0); RotatE 关系=复空间旋转, 能表达对称/反对称/逆/组合。
- **English**: TransE relation = translation (h+r≈t), simple but can't model symmetry (forces r≈0); RotatE relation = complex rotation, expressing symmetry/antisymmetry/inversion/composition.
- **中文**:合成对称关系上 RotatE MRR≈2×TransE、Hits@1 从~0 到~0.56(TransE 数学上无法建模对称); 评估用过滤 MRR/Hits@k。
- **English**: On synthetic symmetric relations RotatE MRR ≈ 2× TransE and Hits@1 goes from ~0 to ~0.56 (TransE mathematically cannot model symmetry); evaluate with filtered MRR/Hits@k.
